# Retrieval Type-B Budget Ablation 정리

이 노트북은 690 chunk 데이터 기준 retrieval 최적화 과정 중, `96` 결과 위에 적용한 target-aware 개선안과 1~5번 추가 실험을 정리한 보고용 노트북입니다.

결과 파일 위치:

- 기준선/target filter 결과: `outputs/retrieval_experiments/target_filter_690_full_v1/`
- 1~5번 ablation 결과: `outputs/retrieval_experiments/typeb_budget_ablation_690_full_v1/`


## 0. 한 줄 결론

최종적으로 가장 성능이 좋았던 조합은 **`D_type_weighted_target_filter` 계열**, 즉 이번 ablation 기준으로는 **`base_D_current`**입니다.

1~5번 개선안을 단독/조합으로 붙여봤지만, 기존 D 세팅보다 nDCG와 multi-doc recall을 더 올린 조합은 없었습니다. 특히 `3. type-B budget target quota`는 여러 정답 문서를 강제로 보존하려는 의도였지만, 실제로는 관련성이 낮은 문서까지 끌어올려 성능이 가장 크게 떨어졌습니다.


## 1. 기준선 비교

먼저 96번 원본 결과와, 그 위에 target-aware 재정렬을 붙인 D 결과를 비교했습니다. 이번 1~5번 실험은 D 결과를 기준선으로 삼았습니다.


| 조합 | 설명 | hit@5 | mrr@5 | ndcg@5 | multi-doc recall@5 | target mismatch rate |
| --- | --- | --- | --- | --- | --- | --- |
| A_current_96_baseline | 96번 원본 retrieval 결과 | 0.9640 | 0.9467 | 0.9255 | 0.9113 | 0.1503 |
| D_type_weighted_target_filter | 96 후보 위에 target-aware 재정렬 적용 | 0.9703 | 0.9467 | 0.9289 | 0.9269 | 0.1071 |
| base_D_current | 이번 1~5번 ablation의 기준선 | 0.9703 | 0.9466 | 0.9288 | 0.9269 | 0.1071 |

해석:

- 96번 원본 대비 D 세팅은 `hit@5`, `nDCG@5`, `multi-doc recall@5`가 모두 좋아졌습니다.
- 특히 `target mismatch rate`가 줄어들어, 비슷한 기관명/사업명 문서가 섞이는 문제가 완화됐습니다.
- 그래서 이후 1~5번 실험은 D 세팅을 더 개선할 수 있는지 확인하는 방식으로 진행했습니다.


## 2. 실험한 개선안

이번에 추가로 확인한 1~5번 개선안은 아래와 같습니다.


| 개선안 | 의미 |
| --- | --- |
| 1. enhanced target extraction | 질문에서 기관명/사업명/공고번호/연도 후보를 조금 더 넓게 추출 |
| 2. similar project suppression | 기관/사업명이 비슷하지만 target과 덜 맞는 문서를 감점 |
| 3. type-B budget target quota | type-B 예산 질문에서 여러 target 문서를 강제로 보존하려는 quota |
| 4. document-then-chunk selection | chunk 점수만 보지 않고 문서 단위 점수를 먼저 만든 뒤 chunk 선택 |
| 5. canonical doc grouping | 문서명 표기 차이를 정규화해서 같은 계열 문서를 묶는 방식 |

## 3. 1~5번 단독 적용 결과

각 개선안을 하나씩만 붙였을 때의 결과입니다. `delta`는 이번 ablation 기준선인 `base_D_current` 대비 변화량입니다.


| 조합 | features | hit@5 | ndcg@5 | multi-doc recall@5 | type-B budget hit@5 | type-B budget ndcg@5 | delta ndcg | delta type-B ndcg |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| base_D_current | base_D | 0.9703 | 0.9288 | 0.9269 | 0.9299 | 0.8806 | 0.0000 | 0.0000 |
| F1_enhanced_target_extraction | 1 | 0.9703 | 0.9288 | 0.9269 | 0.9299 | 0.8806 | 0.0000 | 0.0000 |
| F2_similar_project_suppression | 2 | 0.9693 | 0.9284 | 0.9245 | 0.9271 | 0.8792 | -0.0005 | -0.0013 |
| F3_typeb_budget_target_quota | 3 | 0.9582 | 0.9218 | 0.8970 | 0.8954 | 0.8605 | -0.0071 | -0.0201 |
| F4_doc_then_chunk_selection | 4 | 0.9693 | 0.9284 | 0.9245 | 0.9271 | 0.8793 | -0.0004 | -0.0013 |
| F5_canonical_doc_grouping | 5 | 0.9703 | 0.9288 | 0.9269 | 0.9299 | 0.8806 | 0.0000 | 0.0000 |

단독 실험 해석:

- `1. enhanced target extraction`은 점수 변화가 없었습니다. 이미 기존 D에서 필요한 target 정보가 대부분 잡히고 있었던 것으로 보입니다.
- `5. canonical doc grouping`도 점수 변화가 없었습니다. 현재 top-5 최종 선택에서는 문서명 정규화가 추가 이득으로 이어지지 않았습니다.
- `2. similar project suppression`, `4. document-then-chunk selection`은 아주 조금 하락했습니다. 과한 감점/문서 단위 집계가 일부 정답 문서를 밀어낸 것으로 보입니다.
- `3. type-B budget target quota`는 가장 크게 하락했습니다. 여러 정답 문서를 보존하려는 방향은 맞지만, 강제 quota 방식은 낮은 관련도 문서까지 끌어올릴 위험이 컸습니다.


## 4. 전체 조합 중 상위 결과

1~5번을 가능한 조합으로 붙여본 결과입니다. nDCG 기준 상위 조합만 정리했습니다.


| 조합 | features | hit@5 | ndcg@5 | multi-doc recall@5 | type-B budget hit@5 | type-B budget ndcg@5 | target mismatch rate |
| --- | --- | --- | --- | --- | --- | --- | --- |
| base_D_current | base_D | 0.9703 | 0.9288 | 0.9269 | 0.9299 | 0.8806 | 0.1071 |
| F1_enhanced_target_extraction | 1 | 0.9703 | 0.9288 | 0.9269 | 0.9299 | 0.8806 | 0.1067 |
| F5_canonical_doc_grouping | 5 | 0.9703 | 0.9288 | 0.9269 | 0.9299 | 0.8806 | 0.1083 |
| C15_enhanced_canonical | 1+5 | 0.9703 | 0.9288 | 0.9269 | 0.9299 | 0.8806 | 0.1079 |
| F4_doc_then_chunk_selection | 4 | 0.9693 | 0.9284 | 0.9245 | 0.9271 | 0.8793 | 0.1042 |
| C14_enhanced_doc | 1+4 | 0.9693 | 0.9284 | 0.9245 | 0.9271 | 0.8793 | 0.1034 |
| C45_doc_canonical | 4+5 | 0.9693 | 0.9284 | 0.9245 | 0.9271 | 0.8793 | 0.1051 |
| C145_enhanced_doc_canonical | 1+4+5 | 0.9693 | 0.9284 | 0.9245 | 0.9271 | 0.8793 | 0.1042 |
| F2_similar_project_suppression | 2 | 0.9693 | 0.9284 | 0.9245 | 0.9271 | 0.8792 | 0.1083 |
| C12_enhanced_similar | 1+2 | 0.9693 | 0.9284 | 0.9245 | 0.9271 | 0.8792 | 0.1079 |
| C25_similar_canonical | 2+5 | 0.9693 | 0.9284 | 0.9245 | 0.9271 | 0.8792 | 0.1083 |
| C125_enhanced_similar_canonical | 1+2+5 | 0.9693 | 0.9284 | 0.9245 | 0.9271 | 0.8792 | 0.1079 |

상위 조합 해석:

상위권은 대부분 `base_D_current`, `1`, `5`, `1+5`처럼 실제 ranking을 크게 바꾸지 않는 조합이었습니다. 즉 이번 실험에서 성능을 끌어올린 새 조합은 없었고, 기존 D 세팅이 이미 가장 안정적인 상태였습니다.


## 5. 성능이 크게 떨어진 조합

아래는 nDCG 기준 하위 조합입니다. 대부분 `3. type-B budget target quota`가 포함되어 있습니다.


| 조합 | features | hit@5 | ndcg@5 | multi-doc recall@5 | type-B budget hit@5 | type-B budget ndcg@5 |
| --- | --- | --- | --- | --- | --- | --- |
| C123_enhanced_similar_typeb | 1+2+3 | 0.9515 | 0.9178 | 0.8805 | 0.8764 | 0.8493 |
| C1234_enhanced_similar_typeb_doc | 1+2+3+4 | 0.9515 | 0.9178 | 0.8805 | 0.8764 | 0.8493 |
| C1235_enhanced_similar_typeb_canonical | 1+2+3+5 | 0.9515 | 0.9178 | 0.8805 | 0.8764 | 0.8493 |
| C12345_enhanced_similar_typeb_doc_canonical | 1+2+3+4+5 | 0.9515 | 0.9178 | 0.8805 | 0.8764 | 0.8493 |
| C135_enhanced_typeb_canonical | 1+3+5 | 0.9525 | 0.9183 | 0.8830 | 0.8793 | 0.8506 |
| C1345_enhanced_typeb_doc_canonical | 1+3+4+5 | 0.9525 | 0.9183 | 0.8830 | 0.8793 | 0.8506 |
| C23_similar_typeb | 2+3 | 0.9542 | 0.9194 | 0.8871 | 0.8840 | 0.8537 |
| C235_similar_typeb_canonical | 2+3+5 | 0.9542 | 0.9194 | 0.8871 | 0.8840 | 0.8537 |

하위 조합 해석:

`type-B budget target quota`는 목적 자체는 타당했지만, 현재 구현처럼 정답 후보를 넓히기만 하면 오히려 noise가 늘어납니다. 다음에 다시 시도한다면 quota를 강제로 넣는 방식보다, target 후보 문서의 근거 강도를 더 정확히 계산한 뒤 soft boost를 주는 방식이 좋아 보입니다.


## 6. 실패 사례 미리보기

최적 조합인 `base_D_current`에서도 완전히 해결되지 않은 실패 사례입니다. 주로 여러 정답 문서 중 일부가 빠지거나, 정답 문서가 top-5 안에는 있지만 순위가 낮아 nDCG가 깎이는 경우입니다.


| question_id | hit@5 | ndcg@5 | multi-doc recall@5 | 실패 유형 | 누락 정답 문서 | 검색 top5 |
| --- | --- | --- | --- | --- | --- | --- |
| Q053 |  |  |  |  |  | ["고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf", "고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고.pdf", "고려대학교_고려대학교 공간관리 통합시스템 구축 사업.hwp", "KOICA 전자조달_[지문] 에콰도르 국가 유전자원 데이터 은행 설립 및 역량.hwp", "KOICA 전자조달_[지문] [국제] (재공고)우즈베키스탄 ICT기반의 수자원정보.hwp"] |
| Q054 |  |  |  |  |  | ["한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp", "한국산업기술기획평가원_차세대 NABIS시스템 클라우드 전환·구축 ISP 사.hwp", "한국콘텐츠진흥원_2025~2026년 한국콘텐츠진흥원 정보시스템 통합위탁 운.hwp", "한국의료분쟁조정중재원_차세대 통합정보시스템 구축을 위한 ISP 사업.hwp", "한국기초과학지원연구원_KBSI 차세대 통합정보시스템 구축을 위한 ISMPBPR .hwp"] |
| Q354 |  |  |  |  |  | ["서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf", "서울특별시강동구도시관리공단_(긴급)신규 체육시설 회원관리시스템 구.hwp", "고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고.pdf", "한국교원대학교_한국교원대학교 AI 에듀테크 센터 통합 플랫폼 구축 정.hwp", "서울특별시 서울시립대학교_RISE 사업 평가 대비를 위한 서울시립대학교 .hwp"] |
| Q413 |  |  |  |  |  | ["남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp", "남서울대학교_{대학혁신지원사업] 남서울대학교 대학원 학사시스템 구.hwp", "한국수출입은행_(긴급) SCP 중심의 AML 시스템 구축.pdf", "고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고.pdf", "인천광역시_2024년 IDC 통합 가상화시스템 인프라 고도화.hwp"] |
| Q260 |  |  |  |  |  | ["한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp", "KOICA_가나 북부 2개주 포괄적 일차보건의료체계강화사업 수원국시스템활.hwp", "경상북도 영양군_소규모 공공시설 관리시스템 구축사업.hwp", "해외건설협회_에티오피아 부동산 대량평가 시범시스템 구축 및 전국 확.hwp", "한국수자원공사_건설통합시스템(CMS) 고도화.hwp"] |

## 7. 재현용 코드

아래 셀을 실행하면 원본 CSV에서 결과를 다시 확인할 수 있습니다.


In [ ]:
from pathlib import Path
import pandas as pd

ABLATION_DIR = Path('../outputs/retrieval_experiments/typeb_budget_ablation_690_full_v1')
TARGET_DIR = Path('../outputs/retrieval_experiments/target_filter_690_full_v1')

metrics = pd.read_csv(ABLATION_DIR / 'typeb_budget_ablation_metrics.csv')
target_metrics = pd.read_csv(TARGET_DIR / 'retrieval_target_filter_metrics.csv')
failures = pd.read_csv(ABLATION_DIR / 'best_variant_failure_examples.csv')

metrics.sort_values(['ndcg_at_5', 'hit_at_5', 'multi_doc_recall_at_5'], ascending=False).head(15)


In [ ]:
single_variants = [
    'base_D_current',
    'F1_enhanced_target_extraction',
    'F2_similar_project_suppression',
    'F3_typeb_budget_target_quota',
    'F4_doc_then_chunk_selection',
    'F5_canonical_doc_grouping',
]

cols = [
    'variant_name', 'features', 'hit_at_5', 'ndcg_at_5', 'multi_doc_recall_at_5',
    'focus_hit_at_5', 'focus_ndcg_at_5', 'delta_ndcg_at_5', 'delta_focus_ndcg_at_5',
]
metrics[metrics['variant_name'].isin(single_variants)][cols]


In [ ]:
failure_cols = [
    'question_id', 'question', 'hit_at_5', 'ndcg_at_5', 'multi_doc_recall_at_5',
    'failure_reason', 'ground_truth_docs', 'retrieved_docs_top5', 'missing_gt_docs',
]
failures[failure_cols].head(10)


## 8. 최종 판단

이번 실험 결과만 보면, retrieval 최종 조합은 **`D_type_weighted_target_filter` / `base_D_current`를 유지**하는 것이 가장 좋습니다.

다음 개선 방향은 1~5번을 더 강하게 섞기보다 아래 쪽이 더 가능성이 커 보입니다.

- target 문서 후보를 강제로 넣는 quota가 아니라, 근거 강도 기반 soft boost로 바꾸기
- type-B 질문에서 여러 정답 문서를 더 잘 분리하기 위한 질문 decomposition 개선
- generation 단계에서 top-5만 보는 대신, 검색 후보 중 누락된 target 문서를 추가 확인하는 fallback 설계
